# 🌾 GraoSafe — Análise dos dados de armazenagem

**Disciplina:** Computação Contemporânea e Inteligência Artificial
**Problema local:** em Rondonópolis (MT), calor e umidade dentro do silo causam mofo e perda de grãos.

Neste notebook vamos:
1. Carregar o CSV gerado pelo painel (ou pelo Wokwi);
2. Fazer a **análise exploratória** dos dados;
3. Criar uma **regressão linear** para prever a umidade a partir da temperatura;
4. Concluir em linguagem simples.

## 1. Carregando os dados

In [ ]:
# Bibliotecas que vamos usar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

In [ ]:
# --- Escolha de onde ler o CSV ---
# Opção A (padrão): arquivo local 'exemplo_graosafe.csv' ou 'dados/exemplo_graosafe.csv'
# Opção B: link do Google Drive -> troque o ID abaixo (compartilhe o arquivo como "qualquer pessoa com o link")
LINK_DRIVE = ""   # exemplo: "https://drive.google.com/uc?id=SEU_ID_AQUI"
# Opção C: no Colab, mude para True para abrir a janela de upload
USAR_UPLOAD = False

import os
if LINK_DRIVE:
    caminho = LINK_DRIVE
elif USAR_UPLOAD:
    from google.colab import files   # só existe no Colab
    enviado = files.upload()
    caminho = list(enviado.keys())[0]
else:
    candidatos = ["exemplo_graosafe.csv", "dados/exemplo_graosafe.csv", "../dados/exemplo_graosafe.csv"]
    caminho = next((c for c in candidatos if os.path.exists(c)), None)
    if caminho is None:
        raise FileNotFoundError("CSV não encontrado. Envie o arquivo para o Colab ou use LINK_DRIVE / USAR_UPLOAD.")

df = pd.read_csv(caminho)
print("Lendo:", caminho)
df.head()

In [ ]:
# Preparando as colunas
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["hora_do_dia"] = df["timestamp"].dt.hour + df["timestamp"].dt.minute / 60   # ex.: 14.5 = 14h30
df["risco"] = pd.Categorical(df["risco"], categories=["verde", "amarelo", "vermelho"], ordered=True)
print(df.shape)
df.info()

## 2. Análise exploratória

In [ ]:
# Estatísticas básicas
df[["temperatura", "umidade"]].describe()

In [ ]:
# Quantas leituras em cada nível de risco?
contagem = df["risco"].value_counts().sort_index()
print(contagem)
contagem.plot(kind="bar", color=["#1b7a3a", "#e0a800", "#b3261e"], title="Leituras por nível de risco")
plt.ylabel("Quantidade"); plt.xticks(rotation=0); plt.show()

In [ ]:
# Temperatura e umidade ao longo do tempo, por silo
fig, eixos = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
sns.lineplot(data=df, x="timestamp", y="temperatura", hue="silo", ax=eixos[0], linewidth=1)
eixos[0].set_title("Temperatura por silo"); eixos[0].set_ylabel("°C")
sns.lineplot(data=df, x="timestamp", y="umidade", hue="silo", ax=eixos[1], linewidth=1)
eixos[1].set_title("Umidade por silo"); eixos[1].set_ylabel("% UR")
plt.tight_layout(); plt.show()

In [ ]:
# Histogramas: como os valores se distribuem?
fig, eixos = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=df, x="temperatura", hue="silo", bins=25, ax=eixos[0], element="step")
sns.histplot(data=df, x="umidade", hue="silo", bins=25, ax=eixos[1], element="step")
eixos[0].set_title("Distribuição da temperatura"); eixos[1].set_title("Distribuição da umidade")
plt.tight_layout(); plt.show()

In [ ]:
# Boxplot por silo: mostra mediana e valores fora do normal (pontinhos = episódios de falha)
fig, eixos = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x="silo", y="temperatura", hue="silo", legend=False, ax=eixos[0])
sns.boxplot(data=df, x="silo", y="umidade", hue="silo", legend=False, ax=eixos[1])
eixos[0].set_title("Temperatura por silo"); eixos[1].set_title("Umidade por silo")
plt.tight_layout(); plt.show()

In [ ]:
# Correlação entre as variáveis numéricas
corr = df[["temperatura", "umidade", "hora_do_dia"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Mapa de correlação"); plt.show()

**Como ler:** quanto mais perto de **-1**, mais as duas variáveis andam em sentidos opostos (uma sobe, a outra desce).
Aqui temperatura e umidade têm correlação negativa: quando esquenta, o ar fica mais seco — exceto nos episódios de falha de ventilação, em que as duas sobem juntas.

## 3. Regressão linear: prever a umidade

**Ideia:** dado o valor da temperatura (e a hora do dia), qual a umidade esperada?
Se a umidade real ficar bem **acima** da esperada, algo pode estar errado no silo.

In [ ]:
# Função auxiliar: treina, testa e mostra R² e MAE
def treinar(dados, colunas, nome):
    X, y = dados[colunas], dados["umidade"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)   # 80% treino / 20% teste
    m = LinearRegression().fit(X_tr, y_tr)
    prev = m.predict(X_te)
    print(f"{nome:<38} R² = {r2_score(y_te, prev):.3f} | MAE = {mean_absolute_error(y_te, prev):.2f} pontos de % UR")
    return m, X_te, y_te

# Experimento 1: usando TODAS as leituras (inclui as falhas de ventilação)
treinar(df, ["temperatura"], "Todas as leituras, só temperatura")

# Experimento 2: só operação normal (sem as leituras em vermelho)
normal = df[df["risco"] != "vermelho"]
m_temp, X_te1, y_te1 = treinar(normal, ["temperatura"], "Operação normal, só temperatura")

# Experimento 3: operação normal + hora do dia  (este é o nosso modelo final)
modelo, X_te, y_te = treinar(normal, ["temperatura", "hora_do_dia"], "Operação normal, temperatura + hora")
print("\nCoeficientes:", dict(zip(["temperatura", "hora_do_dia"], modelo.coef_.round(3))), "| intercepto:", round(modelo.intercept_, 2))

**O que aconteceu?** Com todas as leituras o R² é baixo: nas falhas de ventilação temperatura *e* umidade sobem juntas, o que quebra a regra "esquentou → ficou mais seco". Ao treinar só com a operação normal, a reta explica bem a umidade. Ou seja: o modelo descreve o **comportamento esperado** do silo, e o que foge dele é sinal de problema.

In [ ]:
# Gráfico: reta da regressão (só temperatura) sobre a operação normal; as falhas aparecem em vermelho
xs = pd.DataFrame({"temperatura": np.linspace(df["temperatura"].min(), df["temperatura"].max(), 100)})
falhas = df[df["risco"] == "vermelho"]

plt.scatter(normal["temperatura"], normal["umidade"], alpha=0.35, label="Operação normal")
plt.scatter(falhas["temperatura"], falhas["umidade"], alpha=0.6, color="#b3261e", label="Risco vermelho (falhas)")
plt.plot(xs["temperatura"], m_temp.predict(xs), color="black", linewidth=2, label="Reta da regressão")
plt.xlabel("Temperatura (°C)"); plt.ylabel("Umidade (% UR)")
plt.title("Umidade x Temperatura"); plt.legend(); plt.show()

In [ ]:
# Faça sua própria previsão: mude os valores abaixo e rode a célula
temperatura_x = 30.0   # °C
hora_x = 14.0          # hora do dia (14.0 = 14h)

umidade_prevista = modelo.predict(pd.DataFrame({"temperatura": [temperatura_x], "hora_do_dia": [hora_x]}))[0]
print(f"Para {temperatura_x} °C às {hora_x:.0f}h, a umidade esperada é {umidade_prevista:.1f} % UR")
print("Se o sensor medir bem acima disso, vale inspecionar o silo.")

## 4. Conclusão

- **O que vimos:** em dias normais, quando a temperatura sobe, a umidade relativa do ar no silo tende a **cair**. A regressão captura bem essa relação na operação normal (R² alto), mas perde força quando entram os episódios de **falha de ventilação**, em que temperatura *e* umidade sobem juntas — fugindo do padrão.
- **Ligação com o mofo:** fungos crescem quando o grão fica **quente e úmido ao mesmo tempo**. É justamente o que aparece nas leituras de risco *vermelho*. Em Rondonópolis, com calor forte, isso pode acontecer rápido se a ventilação falhar.
- **Uso prático:** um painel como o GraoSafe avisa o produtor assim que o silo sai do padrão, antes de o grão estragar. A regressão pode servir de referência: se a umidade medida ficar muito acima da esperada para aquela temperatura, vale inspecionar o silo.
- **Limitações:** os dados são **simulados** e os limites de risco são **didáticos**. Para uso real seria preciso calibrar com dados de sensores e normas técnicas de armazenagem.